In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama

In [3]:
model = ChatOllama(model="llama3.2")

### Prompt chaining is nothing but using multiple linear llm calls in the langgraph

Here we are doing: topic -> outline of the topic -> detailed report on the topic

In [2]:
# defining the state of the graph
class LLMState(TypedDict):
    title: str
    outline: str
    report: str

In [4]:
# define the generate_outline node
def generate_outline(state: LLMState) -> LLMState:
    title = state["title"]
    prompt = f"Generate a detailed outline for a report with the title: {title}"
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state

In [5]:
# define the generate_report node
def generate_report(state: LLMState) -> LLMState:    
    title = state["title"]
    outline = state["outline"]
    prompt = f"Generate a detailed report based on the following title and outline.\nTitle: {title}\nOutline: {outline}"
    report = model.invoke(prompt).content
    state["report"] = report
    return state

In [7]:
# graph definition
graph = StateGraph(LLMState)

#nodes
graph.add_node("generate_outline", generate_outline)
graph.add_node("generate_report", generate_report)

#edges
graph.add_edge(START, "generate_outline")
graph.add_edge("generate_outline", "generate_report") 
graph.add_edge("generate_report", END)

workflow = graph.compile()

In [8]:
initial_state = LLMState(title="The impact of climate change on global agriculture", outline="", report="")
final_state = workflow.invoke(initial_state)

In [9]:
final_state

{'title': 'The impact of climate change on global agriculture',
 'outline': 'Here is a detailed outline for a report on "The Impact of Climate Change on Global Agriculture":\n\n**I. Introduction**\n\n* Brief overview of the importance of agriculture in the global economy and food security\n* Definition of climate change and its causes\n* Thesis statement: Climate change has significant impacts on global agriculture, affecting crop yields, water availability, and livelihoods of farmers.\n\n**II. Physical Impacts of Climate Change on Agriculture**\n\n* Rising temperatures:\n + Changes in growing seasons and duration\n + Shifts in species composition and distribution\n + Increased frequency and severity of heatwaves and droughts\n* Precipitation patterns:\n + Changes in rainfall intensity and variability\n + Alterations in precipitation schedules and patterns\n + Increased risk of floods and landslides\n* Water scarcity:\n + Reduced water availability due to increased evaporation and runo